# AVAX LONG — thuận xu hướng: Bollinger dài hạn + RSI ngắn hạn + lọc độ rộng dải, train full grid trên GPU

Bản **long** đối xứng của `short_bollinger+RSI+trend` (cùng grid, cùng các cột `max_holding_bars`, `band_ratio_max`). Notebook này dùng `train_full_grid_gpu.py` cùng thư mục (logic chung ở `reference/multi_position/multi_position_engine.py`, luật breakout `entry_mode = 0`). Sửa **ô Cấu hình**, rồi chọn **Run All**. Kết quả lưu thành CSV cạnh notebook.

**Vào lệnh ở close của nến `i`:** `close[i-1] < upper[i] < close[i]` (đóng cửa vượt lên trên dải trên) **và** `RSI[i] > rsi_threshold`. `rsi_threshold = 0` là **tắt RSI** (bản short dùng 100).

**Rủi ro:** risk = `sl_multiplier × (entry − lower[i])` ≈ `sl × độ rộng dải`; stop = entry − risk; TP = entry + `tp_sl_ratio` × risk; `tp_sl_ratio = 0` là **không chốt lời** (chỉ stop hoặc hết thời gian giữ). Timeout `max_holding_bars` nến (cột của grid; `MAX_HOLDING_BARS` chỉ dùng khi grid không có cột này); phí vào lệnh `0.001 / sl_ratio` (bỏ lệnh nếu > 0.2R); phí giữ lệnh mỗi 8 giờ (`FEE_BARS = None` ⇒ 96 ở 5min, 32 ở 15min); một lệnh mỗi lúc.

**Bộ lọc độ rộng dải** (`band_ratio_max`, 0 = tắt): tín hiệu chỉ được vào lệnh khi độ rộng dải lúc đó `(upper − lower) / close` ≤ `band_ratio_max` × **trung vị** độ rộng của 30 ngày trước đó — tức là phá lên từ một dải đang hẹp so với gần đây. 10 ngày đầu của dữ liệu chưa đủ để tính nên không có lệnh.

**Số lệnh mỗi năm** (`min_trades_per_year`, cột kết quả): số lệnh đóng trong **năm ít lệnh nhất**, mỗi năm quy ra cả năm. Đây là **bộ lọc xếp hạng** (`MIN_TRADES_PER_YEAR`), không phải tham số chiến lược.

**Wilson lower bound** và **xếp hạng theo vùng lân cận** (`NEIGHBOUR_RANKING`): như notebook short — xem mô tả ở `short_bollinger+RSI+trend/train_full_grid_gpu.ipynb`.

**ID của trial** là số thứ tự trong grid: đổi `SEARCH_SPACES` là ID đổi theo.

Grid search trên toàn khoảng train là **in-sample**; đánh giá ngoài mẫu bằng `reference/walk_forward_multi_asset.py --folder long_bollinger+RSI+trend_v2` (đọc grid và `SCORE_FORMULA` từ ô Cấu hình này).

In [ ]:
# CẤU HÌNH — sửa ô này trước khi chạy
TIMEFRAME = "5min"                 # ví dụ "5min", "15min", "1h", "4h"
START_DATE = "2025-01-01"          # ngày bắt đầu, bao gồm ngày này
END_DATE = "2026-07-01"            # ngày kết thúc, bao gồm ngày này
DATA_CSV = "../data_5m/avaxusdt_futures_5min.csv"                     # None = reference/data/avaxusdt_futures_5min.csv
SCRIPT_PATH = None                  # None = tự tìm train_full_grid_gpu.py

SCORE_FORMULA = "r_sharpe_monthly"
#SCORE_FORMULA = "r_martin_monthly"
#SCORE_FORMULA = "performance"      # cùng NEIGHBOUR_RANKING = True
#SCORE_FORMULA = "performance/max_dd"
#SCORE_FORMULA = "performance/number_of_trade*r2**5"
#SCORE_FORMULA = "wilson_fitness"
WILSON_CONFIDENCE = 0.9            # cận dưới một phía: 0.90 (z=1.282), 0.95 (z=1.645), 0.99 (z=2.326)
NEIGHBOUR_RANKING = False           # True = xếp hạng theo trung vị score của vùng lân cận
NEIGHBOUR_RADIUS = 1               # số nấc lệch tối đa của mỗi tham số trong danh sách SEARCH_SPACES
# Sửa trực tiếp các danh sách dưới đây để đổi grid. Số bộ tham số là tích số giá trị mỗi danh sách.
# tp_sl_ratio = 0 nghĩa là không chốt lời (chỉ stop hoặc hết thời gian giữ).
# rsi_threshold = 0 nghĩa là tắt RSI (long vào lệnh khi RSI > ngưỡng).
# max_holding_bars: số nến tối đa giữ lệnh (timeout), mỗi bộ một giá trị.
# band_ratio_max: chỉ vào lệnh khi độ rộng dải ≤ band_ratio_max × trung vị độ rộng 30 ngày trước (0 = tắt bộ lọc).
SEARCH_SPACES = {
    "boll_window": [40, 45, 50, 60, 80, 100, 150, 200, 400],
    "boll_window_dev": [1.8, 2.0, 2.2, 2.5],
    #"rsi_window": [5, 7, 9, 14, 20, 30],
    #"rsi_threshold": [60, 65, 70, 75, 80, 85, 90, 95],
    "rsi_window": [14],
    "rsi_threshold": [0],
    "sl_multiplier": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5],
    "tp_sl_ratio": [1.1],
    "max_holding_bars": [192, 240, 288, 336, 384],
    "band_ratio_max": [0.3, 0.5, 0.75, 1.0, 1.5],
}
LIMIT = None                         # None = toàn bộ bộ tham số; số nguyên để chạy thử
MAX_HOLDING_BARS = 288             # chỉ dùng khi SEARCH_SPACES không có max_holding_bars
FEE_BARS = None                    # None = 8 giờ mỗi lần phí (96 ở 5min, 32 ở 15min)
VERIFY_SAMPLES = 0                  # 0 = bỏ qua so với CPU; ví dụ 8 = kiểm tra 8 bộ
MIN_TRADES = 100                     # bảng xếp hạng chỉ giữ bộ có ít nhất số lệnh này (cũng là ngưỡng để tính làm lân cận)
MIN_TRADES_PER_YEAR = 50           # và ít nhất số lệnh này trong năm ít lệnh nhất (quy ra cả năm; 0 = không lọc)
OUTPUT_CSV = None                   # None = CSV theo timeframe, cạnh notebook

In [ ]:
# Nạp script cùng thư mục (cũng hoạt động khi kernel bắt đầu ở thư mục khác trong repo)
from pathlib import Path
import importlib.util
import itertools
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SIDE_FOLDER = "long_bollinger+RSI+trend_v2"
if SCRIPT_PATH is None:
    candidates = []
    for parent in (Path.cwd(), *Path.cwd().parents):
        if parent.name == SIDE_FOLDER:
            candidates.append(parent / "train_full_grid_gpu.py")
        candidates.append(parent / "reference" / SIDE_FOLDER / "train_full_grid_gpu.py")
    script_path = next((path.resolve() for path in candidates if path.is_file()), None)
    if script_path is None:
        raise FileNotFoundError("Không tìm thấy train_full_grid_gpu.py; hãy đặt SCRIPT_PATH ở ô Cấu hình")
else:
    script_path = Path(SCRIPT_PATH).expanduser().resolve()
if not script_path.is_file():
    raise FileNotFoundError(script_path)

# Bỏ engine đã cache trong kernel, để chạy lại ô này là nạp bản engine mới nhất (không cần restart kernel)
sys.modules.pop("multi_position_engine", None)
spec = importlib.util.spec_from_file_location("avax_long_bollinger_RSI_trend_v2", script_path)
strategy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy)
csv_path = Path(DATA_CSV).expanduser().resolve() if DATA_CSV else strategy.DEFAULT_CSV
if not csv_path.is_file():
    raise FileNotFoundError(csv_path)
print("CSV:", csv_path)
print("Đã nạp:", script_path)
print("Phí giữ lệnh: mỗi", strategy.resolve_fee_bars(TIMEFRAME, FEE_BARS), "nến")

In [ ]:
# Đọc dữ liệu, resample và tạo grid
strategy.timeframe_nanos(TIMEFRAME)
start_date, end_date = START_DATE, END_DATE
expected_keys = set(strategy.SEARCH_SPACE)
space = SEARCH_SPACES
if set(space) != expected_keys:
    raise ValueError(f"SEARCH_SPACES cần đúng các khóa: {sorted(expected_keys)}")
for name, values in space.items():
    if not isinstance(values, (list, tuple)) or not values:
        raise ValueError(f"{name} phải là danh sách có ít nhất một giá trị")
space = {name: list(values) for name, values in space.items()}

bars = strategy.load_bars(csv_path, TIMEFRAME, start_date, end_date)
combinations = itertools.product(*(space[name] for name in space))
if LIMIT is not None:
    combinations = itertools.islice(combinations, LIMIT)
grid = pd.DataFrame(combinations, columns=list(space))
print(f"Dữ liệu: {len(bars):,} nến; {bars['timestamp'].iloc[0]} → {bars['timestamp'].iloc[-1]}")
print(f"Grid: {len(grid):,} bộ tham số")
display(grid.head())

In [ ]:
# Chạy toàn bộ grid trên GPU
result, timing = strategy.run_gpu_grid(
    bars, grid, TIMEFRAME,
    max_holding_bars=MAX_HOLDING_BARS,
    fee_bars=FEE_BARS,
    score_formula=SCORE_FORMULA,
)
print("Thời gian:", timing)
print(f"Đã tính {len(result):,} bộ tham số")

In [ ]:
# TÍNH LẠI SCORE — chạy lại riêng ô này sau khi đổi SCORE_FORMULA, WILSON_CONFIDENCE, NEIGHBOUR_* hoặc MIN_TRADES*
result = strategy.add_wilson_columns(result, WILSON_CONFIDENCE)
result = strategy.score_result(result, SCORE_FORMULA)
result["score_own"] = result["score"]  # score của chính bộ tham số
# bộ đủ điều kiện xếp hạng: đủ MIN_TRADES lệnh tổng và đủ MIN_TRADES_PER_YEAR lệnh ở năm ít lệnh nhất
eligible = (result["number_of_trade"] >= MIN_TRADES) & (result["min_trades_per_year"] >= MIN_TRADES_PER_YEAR) #& (result["r2"]>0.96)   #  & (result["win_rate"]<0.57)
if NEIGHBOUR_RANKING:
    # score = trung vị score_own của mọi bộ lệch ≤ NEIGHBOUR_RADIUS nấc ở từng tham số (gồm chính nó); bộ không đủ
    # điều kiện không được tính làm lân cận, và bộ có dưới nửa số lân cận hợp lệ bị bỏ (score = NaN)
    result["score"] = strategy.neighbour_score(result.assign(score_own=result["score_own"].where(eligible)), space,
                                               "score_own", radius=NEIGHBOUR_RADIUS)
print("Score:", SCORE_FORMULA, f"| Wilson một phía {WILSON_CONFIDENCE:.0%}",
      f"| xếp hạng theo trung vị lân cận ±{NEIGHBOUR_RADIUS} nấc" if NEIGHBOUR_RANKING else "| xếp hạng theo score riêng",
      f"| đủ điều kiện: {int(eligible.sum()):,}/{len(result):,} bộ")

In [ ]:
# Kiểm tra CPU (tùy chọn): đối chiếu mẫu với replay float64
if VERIFY_SAMPLES:
    strategy.verify_samples(bars, result, VERIFY_SAMPLES, MAX_HOLDING_BARS, FEE_BARS, TIMEFRAME)

In [ ]:
# Lưu kết quả và xem các bộ tham số đứng đầu
output_path = (Path(OUTPUT_CSV).expanduser().resolve() if OUTPUT_CSV
               else script_path.with_name(f"full_gpu_grid_results_{TIMEFRAME}_fp32.csv"))
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path, index=False)

ranked = result.loc[eligible & result["score"].notna()].sort_values("score", ascending=False)

print("Đã lưu:", output_path)
print(f"Bộ có score hợp lệ, ≥ {MIN_TRADES} lệnh và ≥ {MIN_TRADES_PER_YEAR} lệnh/năm: {len(ranked):,}/{len(result):,}")
display(ranked.head(100))

In [ ]:
# So sánh theo từng chiều tham số: trung vị toàn grid (ít nhiễu hơn bộ "tốt nhất")
view = result.assign(tp_kind=np.where(result["tp_sl_ratio"] > 0, "R cố định", "không TP"))


def family(frame):
    return pd.Series({
        "số bộ": len(frame),
        "lệnh (trung vị)": frame["number_of_trade"].median(),
        f"% bộ ≥ {MIN_TRADES} lệnh": (frame["number_of_trade"] >= MIN_TRADES).mean(),
        "lệnh/năm ít nhất (trung vị)": frame["min_trades_per_year"].median(),
        "R/tháng (trung vị)": frame["r_per_month"].median(),
        "% bộ có lãi": (frame["performance"] > 0).mean(),
        "max DD (trung vị)": frame["max_dd"].median(),
        "win rate (trung vị)": frame["win_rate"].median(),
        "Wilson fitness (trung vị)": frame["wilson_fitness"].median(),
        "% bộ Wilson fitness > 0": (frame["wilson_fitness"] > 0).mean(),
    })


for columns in (["tp_sl_ratio"], ["rsi_window"], ["boll_window"], ["rsi_threshold"], ["sl_multiplier"],
                ["max_holding_bars"], ["band_ratio_max"]):
    print("Theo", ", ".join(columns))
    display(view.groupby(columns).apply(family).round(3))

In [ ]:
# Biểu đồ: phân bố score và hiệu quả theo drawdown
if ranked.empty:
    print("Không có score hợp lệ để vẽ")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    ranked["score"].plot.hist(ax=axes[0], bins=50, title="Phân bố score")
    axes[0].set_xlabel("Score")
    axes[1].scatter(ranked["max_dd"], ranked["performance"], s=5, alpha=0.3)
    axes[1].set(xlabel="max_dd (R)", ylabel="Performance (R)", title="Performance và drawdown")
    plt.tight_layout()
    plt.show()

In [ ]:
# Điền ID của trial trong bảng `result` (đếm từ 0), rồi chạy riêng ô này. None = bộ đứng đầu `ranked`.
# ID là số thứ tự trong grid: đổi SEARCH_SPACES là ID đổi theo. TRIAL_PARAMS tìm theo tham số (thắng TRIAL_ID);
# tham số không ghi trong TRIAL_PARAMS lấy giá trị mặc định bên dưới.
TRIAL_ID = None  # Ví dụ 36240; None = bộ đứng đầu ranked
TRIAL_PARAMS = None  # ví dụ:
#TRIAL_PARAMS = {"boll_window": 400, "boll_window_dev": 2.0, "rsi_window": 14, "rsi_threshold": 0,
#                "sl_multiplier": 0.5, "tp_sl_ratio": 1.1, "max_holding_bars": 384, "band_ratio_max": 0.75}
BACKTEST_START_DATE = "2020-01-01"  # Ví dụ "2025-07-01"; None = ngày bắt đầu train
BACKTEST_END_DATE = "2026-10-14"    # Ví dụ "2026-01-14"; None = ngày kết thúc train

if TRIAL_PARAMS is not None:
    defaults = {"max_holding_bars": MAX_HOLDING_BARS, "band_ratio_max": 0}
    wanted = {**{k: v for k, v in defaults.items() if k in grid}, **TRIAL_PARAMS}
    match = np.ones(len(grid), dtype=bool)
    for name, value in wanted.items():
        match &= np.isclose(grid[name].to_numpy(dtype=float), float(value))
    if not match.any():
        raise KeyError(f"Không có bộ {wanted} trong grid hiện tại")
    TRIAL_ID = int(np.flatnonzero(match)[0])
if TRIAL_ID is None:
    if ranked.empty:
        raise ValueError("Bảng ranked rỗng; hãy điền TRIAL_ID")
    TRIAL_ID = int(ranked.index[0])
if TRIAL_ID not in result.index:
    raise KeyError(f"TRIAL_ID={TRIAL_ID} không có trong result.index")
params = grid.loc[TRIAL_ID].to_dict()
backtest_start = BACKTEST_START_DATE or start_date
backtest_end = BACKTEST_END_DATE or end_date
backtest_bars = (bars if BACKTEST_START_DATE is None and BACKTEST_END_DATE is None
                 else strategy.load_bars(csv_path, TIMEFRAME, backtest_start, backtest_end))

# GPU cho đúng một bộ tham số, và replay float64 liệt kê từng lệnh; hai bên phải khớp.
backtest_result, _ = strategy.run_gpu_grid(
    backtest_bars, grid.loc[[TRIAL_ID]].reset_index(drop=True), TIMEFRAME,
    max_holding_bars=MAX_HOLDING_BARS, fee_bars=FEE_BARS, score_formula=SCORE_FORMULA)
backtest_result = strategy.score_result(strategy.add_wilson_columns(backtest_result, WILSON_CONFIDENCE),
                                        SCORE_FORMULA)
backtest_trial = backtest_result.iloc[0]
trades = strategy.cpu_trace(backtest_bars, params, TIMEFRAME, MAX_HOLDING_BARS, FEE_BARS)
replay = strategy.trade_metrics(backtest_bars, trades, TIMEFRAME, FEE_BARS)
if replay["number_of_trade"] != int(backtest_trial["number_of_trade"]):
    print(f"Lưu ý: GPU {int(backtest_trial['number_of_trade'])} lệnh, replay float64 {replay['number_of_trade']} "
          "(giá sát ngưỡng làm tròn fp32)")

print(f"Trial {TRIAL_ID}. Giai đoạn backtest: {backtest_bars['timestamp'].iloc[0]} đến {backtest_bars['timestamp'].iloc[-1]}")
print("Tham số:", params)
display(backtest_trial.to_frame(name="backtest_trial").T)
display(trades["outcome"].value_counts().to_frame("số lệnh"))
if not trades.empty:
    # Wilson fitness chỉ là expectancy đầy đủ khi payoff gần nhị phân; timeout làm payoff biến thiên
    print(f"Wilson {WILSON_CONFIDENCE:.0%}: win rate {backtest_trial['win_rate']:.2%}, "
          f"p_L {backtest_trial['wilson_p_lower']:.2%}, fitness {backtest_trial['wilson_fitness']:.3f} R/lệnh "
          f"(quan sát {trades['net_r'].mean():.3f} R/lệnh); timeout/hết dữ liệu "
          f"{trades['outcome'].isin(['timeout', 'end_of_data']).mean():.1%} số lệnh")
    by_year = trades.groupby(trades["close_timestamp"].dt.year)["net_r"].agg(["size", "sum"])
    display(by_year.rename(columns={"size": "số lệnh", "sum": "R"}).round(2).T)
display(trades.head(20))

if trades.empty:
    print("Không có lệnh nào trong khoảng backtest")
else:
    equity = strategy.mtm_equity(backtest_bars, trades, TIMEFRAME, FEE_BARS)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(equity["timestamp"], equity["mtm_equity"], color="lightgreen", linewidth=0.8, label="MTM (gồm lệnh đang mở)")
    ax.plot(equity["timestamp"], equity["closed_equity"], color="green", label="Lệnh đã đóng")
    ax.set(xlabel="Time", ylabel="Cumulative R (sau phí)",
           title=f"AVAX LONG thuận xu hướng — trial {TRIAL_ID} ({backtest_start} to {backtest_end})")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    plt.show()

In [ ]:
ax = backtest_bars.set_index("timestamp")["close"].plot(figsize=(18, 6))
ax.set(title="AVAX/USDT close", xlabel="Time", ylabel="Price (USDT)")
plt.show()